# tool_eval — 도구 개수·체인 깊이가 정확도에 미치는 영향 (multi-turn A/B)

**가설:** 도구가 2개·2스텝인 단순 케이스에선 통합/분리 정확도가 같았다(둘 다 100%). 하지만
**도구가 많고 체인이 깊은(멀티턴)** 상황에선 분리(프리미티브 다수)가 중간 결과를 여러 번 손으로
이어야 해서 **정확도가 떨어질 것**이다. 이걸 딥한(결정적) 계산으로 검증한다.

| 설계안 | 도구 | 방식 |
|---|---|---|
| **분리(primitive)** | `add/subtract/multiply/divide/power/sqrt/modulo/negate` — **8개** | 식을 프리미티브 호출 시퀀스로 분해해 **여러 턴**에 걸쳐 중간값 threading |
| **통합(calc)** | `calc(expression)` — **1개** | 식 전체를 결정적 평가기로 **1콜** |

식의 깊이(연산자 수)를 2→7 로 키우며 정확도가 갈리는지 본다. 정답은 안전한 AST 평가기로 계산
(검증기·통합도구 공용) → 정확 채점. 턴 수(LLM 호출 수)도 함께 기록해 '멀티턴'을 드러낸다.

## 0. 셋업

In [ ]:
import os, sys, json, time, re, ast, math, operator, pathlib
from dataclasses import dataclass, field
from typing import Callable

try:
    from dotenv import load_dotenv
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (base / ".env").exists():
            load_dotenv(base / ".env"); break
except Exception:
    pass
from openai import OpenAI
MODEL = "gpt-5-nano"
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
print("OpenAI:", "준비됨 (실행 셀 사용 가능)" if client else "키 없음 — 실행 셀은 건너뜀")

## 딥 엔진(안전한 AST 평가기) + 두 설계안

`calc_expr` 은 ast 화이트리스트(+,-,*,/,**,%, 단항 -, sqrt)만 허용하는 안전 평가기 — `eval` 안 씀.
검증기와 통합 도구가 이걸 공유한다. 분리 도구는 각 연산의 프리미티브.

In [ ]:
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg}

def _ev(node):
    if isinstance(node, ast.Expression): return _ev(node.body)
    if isinstance(node, ast.Constant):   return node.value
    if isinstance(node, ast.BinOp):      return _OPS[type(node.op)](_ev(node.left), _ev(node.right))
    if isinstance(node, ast.UnaryOp):    return _OPS[type(node.op)](_ev(node.operand))
    if isinstance(node, ast.Call) and getattr(node.func, "id", None) == "sqrt":
        return math.sqrt(_ev(node.args[0]))
    raise ValueError("허용되지 않은 식")

def calc_expr(expr):
    return _ev(ast.parse(expr, mode="eval"))

def _num(x):
    # 정수값이면 int 로 (표시·비교 깔끔하게)
    return int(x) if float(x).is_integer() else round(float(x), 6)

# ── 분리안: 프리미티브 8종 ──
def _p_add(a, b): return _num(a + b)
def _p_sub(a, b): return _num(a - b)
def _p_mul(a, b): return _num(a * b)
def _p_div(a, b): return _num(a / b)
def _p_pow(base, exp): return _num(base ** exp)
def _p_sqrt(x): return _num(math.sqrt(x))
def _p_mod(a, b): return _num(a % b)
def _p_neg(x): return _num(-x)

_PRIM = {"add": _p_add, "subtract": _p_sub, "multiply": _p_mul, "divide": _p_div,
         "power": _p_pow, "sqrt": _p_sqrt, "modulo": _p_mod, "negate": _p_neg}

def prim_dispatch(name, args):
    fn = _PRIM.get(name)
    if fn is None:
        return f"<error>알 수 없는 도구: {name}</error>"
    try:
        return str(fn(**args))
    except ZeroDivisionError:
        return "<error>0으로 나눔</error>"
    except TypeError as e:
        return f"<error>인자 오류({name}): {e}</error>"

def calc_dispatch(name, args):
    if name != "calc":
        return f"<error>알 수 없는 도구: {name}</error>"
    try:
        return str(_num(calc_expr(args.get("expression", ""))))
    except Exception as e:
        return f"<error>식 오류: {e}</error>"

def _two(da, db):
    return {"type": "object",
            "properties": {"a": {"type": "number", "description": da}, "b": {"type": "number", "description": db}},
            "required": ["a", "b"]}

def _one(d):
    return {"type": "object", "properties": {"x": {"type": "number", "description": d}}, "required": ["x"]}

TOOLS_PRIMITIVE = [
    {"type": "function", "name": "add",      "description": "두 수를 더한다: a + b",   "parameters": _two("피연산자 a", "피연산자 b")},
    {"type": "function", "name": "subtract", "description": "뺀다: a - b",            "parameters": _two("피감수 a", "감수 b")},
    {"type": "function", "name": "multiply", "description": "곱한다: a * b",           "parameters": _two("피연산자 a", "피연산자 b")},
    {"type": "function", "name": "divide",   "description": "나눈다: a / b",           "parameters": _two("피제수 a", "제수 b")},
    {"type": "function", "name": "power",    "description": "거듭제곱: base ** exp",
     "parameters": {"type": "object", "properties": {"base": {"type": "number", "description": "밑"}, "exp": {"type": "number", "description": "지수"}}, "required": ["base", "exp"]}},
    {"type": "function", "name": "sqrt",     "description": "제곱근: sqrt(x)",         "parameters": _one("피연산자")},
    {"type": "function", "name": "modulo",   "description": "나머지: a % b",           "parameters": _two("피제수 a", "제수 b")},
    {"type": "function", "name": "negate",   "description": "부호 반전: -x",           "parameters": _one("피연산자")},
]

TOOLS_CALC = [
    {"type": "function", "name": "calc",
     "description": "산술식 전체를 한 번에 계산한다. +,-,*,/,**,%, 단항 -, sqrt(x), 괄호 지원. 예: '(312*47)-(5049/3)'",
     "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "평가할 산술식 문자열"}}, "required": ["expression"]}},
]

print("엔진 준비. 예) calc_expr('(312*47)-(5049/3)') =", _num(calc_expr("(312*47)-(5049/3)")))

## 평가 과제 — 깊이 2→7 (Stage 1)

같은 산술식을 두 설계안에 준다. 깊이(연산자 수)가 커질수록 분리안은 더 많은 프리미티브를 여러 턴에
걸쳐 이어야 한다. 검증기는 안전 평가기로 정답을 계산해 `<answer>` 안의 숫자와 비교(허용오차).

In [ ]:
@dataclass
class Task:
    id: str
    split: str
    depth: int
    expr: str

    @property
    def prompt(self):
        return f"다음 산술식을 반드시 도구로 계산해 최종 값을 구하라(직접 암산 금지): {self.expr}"

    def verify(self, answer):
        expected = _num(calc_expr(self.expr))
        i, j = (answer or "").find("<answer>"), (answer or "").find("</answer>")
        seg = answer[i + 8:j] if (i >= 0 and j > i) else (answer or "")
        nums = [float(x) for x in re.findall(r"-?\d+\.?\d*(?:[eE][+-]?\d+)?", seg.replace(",", ""))]  # 콤마 천단위·지수표기 허용
        ok = any(abs(g - expected) <= max(0.001, abs(expected) * 1e-6) for g in nums)
        return ok, f"기대 {expected}"

EVAL_TASKS = [
    Task("E1", "train",   2, "1487 + 2394 - 876"),
    Task("E2", "train",   3, "(312 * 47) - (5049 / 3)"),
    Task("E3", "train",   4, "(9856 - 3721) * 4 + 2048 / 8"),
    Task("E4", "train",   5, "sqrt(1764) * 33 - 12048 / 6 + 512"),
    Task("E5", "heldout", 6, "(4500 / 12 + 875) * 3 - 999 + 2 ** 10"),
    Task("E6", "heldout", 7, "((18500 - 4325) / 25 + 1200) * 4 - sqrt(10000) + 3 ** 5"),
]
print("과제(id, depth, 정답):", [(t.id, t.depth, _num(calc_expr(t.expr))) for t in EVAL_TASKS])

## 실행 하네스 — 턴 수도 센다 (Stage 2)

멀티턴을 드러내기 위해 LLM 호출 횟수(turns)를 함께 기록한다. 분리안은 중간값 의존성 때문에
스텝이 여러 턴으로 이어진다.

In [ ]:
SYSTEM_PROMPT = (
    "너는 계산 에이전트다. 주어진 도구만으로 산술식을 계산하라. 절대 암산으로 답하지 말고 "
    "모든 연산을 도구로 수행하라(중간값도 도구 결과를 그대로 사용).\n"
    "1) 도구 호출 앞에 <plan>...</plan> 으로 다음 스텝을 한 줄 적어라.\n"
    "2) 마지막에 <answer>숫자</answer> 로 최종 값만, 이어서 <tool_feedback>...</tool_feedback> 로 "
    "도구 구성(개수·분해 부담)에 대한 소감을 한 줄 남겨라.\n"
)

def run_task(client, task, tools, dispatch, arm="", model=None, max_turns=25):
    model = model or MODEL
    input_list = [{"role": "user", "content": task.prompt}]
    transcript = []
    n_calls = n_errors = in_tok = out_tok = turns = 0
    final_text = ""
    t0 = time.time()
    for _ in range(max_turns):
        resp = client.responses.create(model=model, instructions=SYSTEM_PROMPT,
                                       input=input_list, tools=tools, parallel_tool_calls=True)
        turns += 1
        if resp.usage:
            in_tok += resp.usage.input_tokens
            out_tok += resp.usage.output_tokens
        input_list += resp.output
        calls = [it for it in resp.output if it.type == "function_call"]
        if not calls:
            final_text = resp.output_text
            break
        for c in calls:
            try:
                args = json.loads(c.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            try:
                result = dispatch(c.name, args)
            except Exception as e:
                result = f"<error>{type(e).__name__}: {e}</error>"
            n_calls += 1
            if result.startswith("<error>"):
                n_errors += 1
            transcript.append((c.name, json.dumps(args, ensure_ascii=False), result[:80]))
            input_list.append({"type": "function_call_output", "call_id": c.call_id, "output": result})
    else:
        final_text = final_text or "(max_turns 도달)"
    passed, reason = task.verify(final_text)
    return {"arm": arm, "task_id": task.id, "depth": task.depth, "passed": passed, "reason": reason,
            "turns": turns, "tool_calls": n_calls, "tool_errors": n_errors,
            "input_tokens": in_tok, "output_tokens": out_tok, "duration_s": round(time.time() - t0, 2),
            "final_text": final_text, "transcript": transcript}

## A/B 실행 — 깊이별 정확도 비교 (Stage 2·3)

두 설계안 × 6깊이 × TRIALS. 분리안은 turns·호출이 깊이에 따라 늘고, 정확도가 갈리는지 본다.
(멀티턴이라 분리안은 느리다 — 토큰·시간 주의)

In [ ]:
ARMS = [("분리(프리미티브 8도구)", TOOLS_PRIMITIVE, prim_dispatch),
        ("통합(calc 1도구)",       TOOLS_CALC,      calc_dispatch)]
TRIALS = 2

if client:
    all_results = []
    for label, tools, disp in ARMS:
        rs = [run_task(client, t, tools, disp, arm=label) for t in EVAL_TASKS for _ in range(TRIALS)]
        all_results += rs
        n = len(rs)
        print(f"{label:22} 정확도 {sum(r['passed'] for r in rs)/n:4.0%} · 평균 turns {sum(r['turns'] for r in rs)/n:.1f} · "
              f"평균 호출 {sum(r['tool_calls'] for r in rs)/n:.1f} · 오류 {sum(r['tool_errors'] for r in rs)} · "
              f"평균 토큰 {sum(r['input_tokens']+r['output_tokens'] for r in rs)/n:,.0f}")
    print("\n깊이별 정확도 (분리 vs 통합):")
    for t in EVAL_TASKS:
        a = [r for r in all_results if r['arm'] == ARMS[0][0] and r['task_id'] == t.id]
        b = [r for r in all_results if r['arm'] == ARMS[1][0] and r['task_id'] == t.id]
        print(f"  {t.id} depth {t.depth} | 분리 {sum(r['passed'] for r in a)}/{len(a)} (turns~{sum(r['turns'] for r in a)/len(a):.0f}, 호출~{sum(r['tool_calls'] for r in a)/len(a):.0f}) | 통합 {sum(r['passed'] for r in b)}/{len(b)}")
    outdir = pathlib.Path.cwd() / "results"; outdir.mkdir(exist_ok=True)
    outfile = outdir / f"chaindepth-{time.strftime('%Y%m%d-%H%M%S')}.jsonl"
    outfile.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in all_results))
    print(f"\nTRIALS={TRIALS} · 저장:", outfile)
else:
    print("OPENAI_API_KEY 없음 — 이 셀 건너뜀")

## 구조 분석 (API 불필요)

In [ ]:
print("분리안 도구 수:", len(TOOLS_PRIMITIVE), "· 통합안 도구 수:", len(TOOLS_CALC))
print("깊이별 식·정답 (분리안이 필요로 하는 최소 프리미티브 호출 수 ~= 연산자 수):")
for t in EVAL_TASKS:
    print(f"  {t.id} depth={t.depth}  {t.expr}  = {_num(calc_expr(t.expr))}")
# 결정적 정합성: 통합 평가기 vs 수동 프리미티브 조립
s = _num(calc_expr("(9856 - 3721) * 4 + 2048 / 8"))
manual = _p_add(_p_mul(_p_sub(9856, 3721), 4), _p_div(2048, 8))
print("통합=수동 프리미티브 동일:", s == manual, "(", s, ")")

## 결론 — 언제 정확도가 갈리나

- **단순(2도구·2스텝)**: 정확도 동률(앞 노트북에서 확인). 통합은 호출·토큰·지연만 이득.
- **다도구·멀티턴(이 노트북)**: 분리안은 깊이에 비례해 turns·호출이 늘고, **중간값 이어붙이기(threading)·도구 선택 실수**가 누적된다. 통합(`calc`)은 결정적 평가기로 깊이와 무관하게 1콜.
- 따라서 **정확도 격차는 '많은 도구 + 깊은 체인'에서 드러난다** — 위 깊이별 표에서 분리안 pass율이 깊이가 커질수록 떨어지면 가설이 맞은 것이다(떨어지지 않으면 그 모델이 그만큼 견고한 것 — 그것도 데이터로 확인된 사실).

**핵심 원리(Anthropic 가이드):** 결정적이고 자주 함께 쓰이는 다단계 로직은 **하나의 도구로 통합**하라.
모델의 오케스트레이션 부담(멀티턴 threading·도구 선택)을 도구 안으로 옮기면 토큰뿐 아니라 **정확도**도 지킨다.
반대로 각 프리미티브가 **독립적으로도 자주 유용**하면 분리의 유연성이 필요하다 — 그땐 통합 대신
'자주 나오는 조합만' 상위 도구로 추가하는 절충이 답이다.